<a href="https://colab.research.google.com/github/409raja/AI-Voice-Assistant-for-Personal-Use/blob/main/Image_to_HTML_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which 

In [2]:
import torch
from transformers import VisionEncoderDecoderModel, GPT2Tokenizer, ViTFeatureExtractor
from datasets import load_dataset
import itertools

# Step 1: Load a small subset of WebSight dataset
def load_subset(dataset_name="HuggingFaceM4/WebSight", num_samples=20):
    dataset = load_dataset(dataset_name, split="train", streaming=True)
    subset = list(itertools.islice(dataset, num_samples))
    return subset

# Step 2: Preprocess images & tokenize HTML
def preprocess_data(subset, tokenizer, feature_extractor):
    inputs = []
    labels = []

    for sample in subset:
        try:
            image = feature_extractor(sample["image"], return_tensors="pt").pixel_values.squeeze(0)

            # Fix: Check available keys
            html_key = "html" if "html" in sample else list(sample.keys())[-1]  # Pick last key if unknown
            html_tokens = tokenizer(sample[html_key], padding="max_length", truncation=True, max_length=512, return_tensors="pt").input_ids.squeeze(0)

            inputs.append(image)
            labels.append(html_tokens)
        except KeyError as e:
            print(f"Skipping sample due to missing key: {e}")

    return torch.stack(inputs), torch.stack(labels)


# Step 3: Load the tokenizer and feature extractor
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
feature_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224")

# ✅ Fix: Set pad token before model initialization
tokenizer.pad_token = tokenizer.eos_token  # or '[PAD]' if you want a separate padding token

# Step 4: Define Model (Vision to Text)
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained("google/vit-base-patch16-224", "gpt2")

# ✅ Fix: Set pad_token_id immediately after model initialization
model.config.pad_token_id = tokenizer.pad_token_id
model.config.decoder_start_token_id = tokenizer.eos_token_id

# Step 5: Load and preprocess the data
subset = load_subset()
inputs, labels = preprocess_data(subset, tokenizer, feature_extractor)

# Step 6: Define Training Configuration
training_args = {
    "epochs": 3,
    "batch_size": 2,
    "learning_rate": 5e-5,
}

optimizer = torch.optim.AdamW(model.parameters(), lr=training_args["learning_rate"])

# Step 7: Train the Model


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Some weights of GPT2LMHeadModel were not initialized from the model checkpoint at gpt2 and are newly initialized: ['h.0.crossattention.c_attn.bias', 'h.0.crossattention.c_attn.weight', 'h.0.crossattention.c_proj.bias', 'h.0.crossattention.c_proj.weight', 'h.0.crossattention.q_attn.bias', 'h.0.crossattention.q_attn.weight', 'h.0.ln_cross_attn.bias', 'h.0.ln_cross_attn.weight', 'h.1.crossattention.c_attn.bias', 'h.1.crossattention.c_attn.weight', 'h.1.crossattention.c_proj.bias', 'h.1.crossattention.c_proj.weight', 'h.1.crossattention.q_attn.bias', 'h.1.crossattention.q_attn.weight', 'h.1.ln_cross_attn.bias', 'h.1.ln_cross_attn.weight', 'h.10.crossattention.c_attn.bias', 'h.10.crossattention.c_attn.weight', 'h.10.crossattention.c_proj.bias', 'h.10.crossattention.c_proj.weight', 'h.10.crossattention.q_attn.bias', 'h.10.crossattention.q_attn.weight', 'h.10.ln_cross_attn.bias', 'h.10.ln_cross_attn.weight', 'h.11.crossattention.c_attn.bias', 'h.11.crossattention.c_attn.weight', 'h.11.crossat

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/738 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/738 [00:00<?, ?it/s]

In [3]:
model.train()
for epoch in range(training_args["epochs"]):
    for i in range(0, len(inputs), training_args["batch_size"]):
        batch_inputs = inputs[i:i+training_args["batch_size"]]
        batch_labels = labels[i:i+training_args["batch_size"]]

        outputs = model(pixel_values=batch_inputs, labels=batch_labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch+1}, Step {i}, Loss: {loss.item()}")

# Step 8: Save Model
model.save_pretrained("image_to_html_model")
tokenizer.save_pretrained("image_to_html_model")
print("Model saved successfully!")

# Step 9: Evaluate on a Test Sample
def generate_html(image):
    image_tensor = feature_extractor(image, return_tensors="pt").pixel_values
    generated_ids = model.generate(pixel_values=image_tensor, max_length=512)
    html_output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    return html_output

# Test the Model on the First Sample
sample_image = subset[0]["image"]
generated_html = generate_html(sample_image)
print("Generated HTML:\n", generated_html)


Epoch 1, Step 0, Loss: 11.365750312805176
Epoch 1, Step 2, Loss: 2.4290478229522705
Epoch 1, Step 4, Loss: 0.6650646328926086
Epoch 1, Step 6, Loss: 0.5142967700958252
Epoch 1, Step 8, Loss: 0.5844423174858093
Epoch 1, Step 10, Loss: 0.4018016457557678
Epoch 1, Step 12, Loss: 0.4562695622444153
Epoch 1, Step 14, Loss: 0.44856560230255127
Epoch 1, Step 16, Loss: 0.5480791330337524
Epoch 1, Step 18, Loss: 0.4920032024383545
Epoch 2, Step 0, Loss: 0.4247116446495056
Epoch 2, Step 2, Loss: 0.42907285690307617
Epoch 2, Step 4, Loss: 0.46263107657432556
Epoch 2, Step 6, Loss: 0.3953363299369812
Epoch 2, Step 8, Loss: 0.4958684742450714
Epoch 2, Step 10, Loss: 0.34822237491607666
Epoch 2, Step 12, Loss: 0.40154406428337097
Epoch 2, Step 14, Loss: 0.3957752585411072
Epoch 2, Step 16, Loss: 0.47659599781036377
Epoch 2, Step 18, Loss: 0.4236525595188141
Epoch 3, Step 0, Loss: 0.35109350085258484
Epoch 3, Step 2, Loss: 0.36918652057647705
Epoch 3, Step 4, Loss: 0.40573886036872864
Epoch 3, Step 6

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model saved successfully!
Generated HTML:
 


In [ ]:
pip install transformers datasets torch pillow
